# DACON MERT12 full-data A100 refit
Frozen MERT layer 12, three frozen-feature heads, every audited eligible row. Former splits are all training data, so this notebook makes no independent-validation claim.

In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile
JOB = Path('/root/mert12_a100_20260918')
SOURCE, INPUT = JOB/'source', JOB/'input'
OUTPUT = Path('/root-1/mert12-a100-20260918/output')
OUTPUT.mkdir(parents=True, exist_ok=True)
required = [SOURCE/'mert12_full_refit.py', INPUT/'full_data_manifest.csv', INPUT/'joined_data_audit.json', INPUT/'features', INPUT/'assets', INPUT/'smoke_inputs.zip']
assert all(path.exists() for path in required), [str(path) for path in required if not path.exists()]

In [ ]:
subprocess.run([sys.executable, str(SOURCE/'mert12_full_refit.py'), '--manifest', str(INPUT/'full_data_manifest.csv'), '--features', str(INPUT/'features'), '--audit-receipt', str(INPUT/'joined_data_audit.json'), '--output', str(OUTPUT/'mert12_full'), '--device', 'cuda', '--benchmark-batches'], check=True, cwd=SOURCE)

In [ ]:
subprocess.run([sys.executable, str(SOURCE/'package_mert12_submission.py'), '--root', str(SOURCE), '--assets', str(INPUT/'assets'), '--run', str(OUTPUT/'mert12_full'), '--output', str(OUTPUT/'MERT12_full_submit.zip')], check=True, cwd=SOURCE)

In [ ]:
SMOKE_PACKAGE, SMOKE_INPUTS = JOB/'smoke_package', JOB/'smoke_inputs'
assert not SMOKE_PACKAGE.exists() and not SMOKE_INPUTS.exists(), 'fresh smoke directories required'
with zipfile.ZipFile(OUTPUT/'MERT12_full_submit.zip') as archive: archive.extractall(SMOKE_PACKAGE)
with zipfile.ZipFile(INPUT/'smoke_inputs.zip') as archive: archive.extractall(SMOKE_INPUTS)
subprocess.run([sys.executable, str(SOURCE/'mert12_submission.py'), 'cached-reference', '--features', str(INPUT/'features'), '--checkpoints', *[str(OUTPUT/f'mert12_full/head_seed{seed}.pt') for seed in (41, 42, 43)], '--sample-submission', str(SMOKE_INPUTS/'original/sample_submission.csv'), '--output', str(OUTPUT/'cached_mert12_reference.json')], check=True, cwd=SOURCE)
for name in ('original', 'permuted', 'subset'):
    target = OUTPUT/f'smoke_{name}'
    target.mkdir()
    subprocess.run([sys.executable, str(SMOKE_PACKAGE/'script.py'), '--test-dir', str(SMOKE_INPUTS/name/'test'), '--sample-submission', str(SMOKE_INPUTS/name/'sample_submission.csv'), '--output', str(target/'submission.csv'), '--device', 'cuda'], check=True)
subprocess.run([sys.executable, str(SOURCE/'mert12_submission.py'), 'compare', '--reference', str(OUTPUT/'smoke_original/submission.csv'), '--permuted', str(OUTPUT/'smoke_permuted/submission.csv'), '--subset', str(OUTPUT/'smoke_subset/submission.csv'), '--cached-reference', str(OUTPUT/'cached_mert12_reference.json'), '--output', str(OUTPUT/'submission_smoke.json')], check=True, cwd=SOURCE)

In [ ]:
receipt = json.loads((OUTPUT/'mert12_full/full_refit_receipt.json').read_text())
package = json.loads((OUTPUT/'MERT12_full_submit.json').read_text())
smoke = json.loads((OUTPUT/'submission_smoke.json').read_text())
{'status': 'complete', 'training_rows': receipt['eligible_rows'], 'seeds': receipt['seeds'], 'batch_size': receipt['batch_size'], 'zip_sha256': package['sha256'], 'zip_bytes': package['bytes'], 'smoke': smoke, 'independent_validation': receipt['independent_validation']}